---
title: Error correction
description: Explore quantum error correction, decoding, and fault tolerant architectures through their resource costs and role in quantum advantage.
---

{/* cspell:ignore Bartolucci foldables leftrightarrow anticommuted bivariate transmons zorder xytext arrowprops arrowstyle LDPC Terhal FTQC biplanar buildable roadmaps Yoder Hypergraph transversality Toffolis triorthogonal codespace ASIC ASICs MWPM affordances teraquop PASQAL Zémor Panteleev Kalachev Mariantoni Cleland Bombín anyons Horsman Devitt Haah Shutty Müller Maurer */}

# Quantum error correction: when is it the right tool for quantum advantage?

<Admonition type="note" title="Prerequisites">

This lesson builds on the stabilizer-check language introduced in the error detection lesson. This module assumes you are comfortable with Pauli checks, syndromes, code distance, and the idea that surface-code checks are local on a 2D layout. A full formal treatment of stabilizer codes is available in John Watrous's [Foundations of Quantum Error Correction](/learning/courses/foundations-of-quantum-error-correction/) course on IBM Quantum Learning, but the main text only requires the working ideas summarized in the optional refresher directly below. We stay at the level of *using* the formalism rather than deriving it.

</Admonition>

<Accordion>
<AccordionItem title="New to stabilizers, or want a quick review? Open this 5-minute refresher.">

The refresher below is *optional* and only reviews the minimum notation needed for correction and decoding — enough to follow Parts 1–6 (the foldables and Section 3.1 details assume more):

- **Pauli operators.** $X$ flips $|0\rangle \leftrightarrow |1\rangle$ (bit flip); $Z$ flips the sign of $|1\rangle$ (phase flip); $Y = iXZ$.
- **Stabilizer / check.** A multi-qubit Pauli (e.g. $Z_1 Z_0$) that encoded states leave unchanged (eigenvalue $+1$). Measuring it is a *parity check*: $+1$ if undisturbed, $-1$ if an error anticommuted with it — *without* measuring the encoded data itself.
- **Syndrome.** The list of $\pm 1$ check outcomes — a classical bit string carrying *partial* information about the error (not its exact location, and nothing about the encoded state). A decoder infers the most likely error from it.
- **Weight.** The number of qubits an error (a multi-qubit Pauli) acts on nontrivially — e.g. $Z_4X_1$ has weight 2.
- **Code distance $d$.** The weight of the smallest error that can slip past all checks as a *logical* error. Larger $d$ = more protection.
- **$[[n, k, d]]$.** $n$ physical qubits protect $k$ logical qubits with distance $d$. Example: $[[7,1,3]]$ protects 1 logical qubit in 7 physical, distance 3.
- **CSS code.** Checks split into pure-$X$ and pure-$Z$ types handled independently. The surface code and BB codes are CSS.

That is the whole vocabulary the main text leans on. Section 1.2 then lets you *run* a complete cycle on the smallest example.

</AccordionItem>
</Accordion>

This lesson asks not _what_ quantum error correction is, but _when it is the right tool for quantum advantage_. To answer that, we track where the cost comes from (section 1–2), how it can be reduced (section 3), and where it *moves* rather than vanishes (section 4–5) — then fold all of it into the advantage judgment (section 6).

The earlier lessons introduced several ways to keep quantum computations accurate. We saw how physical errors arise and accumulate, how they can sometimes be suppressed or reshaped, and how their effect can be mitigated through additional runs and classical post-processing. The previous lesson added another idea: build checks into the computation so that a corrupted run can be identified and discarded. Its code-based syndrome post-selection, including spacetime checks, is a form of post-selected quantum error correction ([Qiskit Paulice blog](https://www.ibm.com/quantum/blog/qiskit-paulice)). This improves the quality of the retained data, but it also leaves limitations. Some errors pass undetected; a triggered check may reveal that something went wrong without identifying the most likely fault; and the method is most effective for circuits whose structure supports low-overhead checks. Decoding-based correction builds on the same check information: instead of asking whether a run should be kept or discarded, it asks which error is most consistent with the observed pattern and how the encoded information should be recovered. The distinction here is between syndrome-based post-selection and decoding with recovery, two approaches along the same QEC continuum ([QEC continuum blog](https://www.ibm.com/quantum/blog/qec-continuum)). That extra power changes the resource tradeoff: decoding-based correction shifts more of the cost toward physical qubits, syndrome extraction, decoding, connectivity, and architecture. This trades sampling cost for additional qubits and hardware resources along the QEC continuum ([QEC continuum blog](https://www.ibm.com/quantum/blog/qec-continuum)), becoming central when lighter-weight methods cannot support the depth or reliability a task requires.

This module begins by connecting decoding and recovery to the post-selection methods of the previous module, then turns to the most established way to protect stored quantum information and why it is so expensive. From there it asks how that cost might be reduced and examines a concrete recent proposal as a case study. It then explains why protecting stored information is not enough for useful computation and what additional resources a full algorithm needs, before stepping back to view error correction as a whole system — decoders, modular architecture, and how other hardware platforms inherit different challenges — and closing on the central question of Lanes et al.: when does error correction help preserve a quantum advantage, and when does its overhead erase it?

<Admonition type="note" title="A note on scope">

Quantum error correction is a problem for every hardware platform, but the form it takes differs. This lesson uses recent IBM work on bivariate-bicycle and gross-code architectures as a concrete case study, while keeping the main framing from [Lanes et al., 2025](https://arxiv.org/abs/2506.20658). Within that scope we focus on *stabilizer codes on 2D superconducting transmons* (a *transmon* is the most common type of superconducting qubit), where the design challenge is to lay out checks on a flat grid and absorb non-local connectivity through modular hardware. Trapped-ion, neutral-atom, bosonic-mode, and topological-qubit platforms inherit different challenges; we return to them in Part 5. Until then, "physical qubit", "stabilizer check", and "2D layout" should be read as describing the superconducting setting.

</Admonition>

We begin by importing the libraries used throughout the lesson.

In [1]:
# Shared imports and setup

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

np.random.seed(0)


# Global matplotlib setting

colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

plt.rcParams.update(
    {
        "figure.figsize": (6, 4),
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": False,
        "axes.labelsize": 11,
        "axes.titlesize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.frameon": False,
        "legend.fontsize": 10,
    }
)


# Surface-code toy fit (Fowler and Gidney) and patch qubit count (Bravyi et al.).
def surface_logical_error(d, p=1e-3, p_th=1e-2, A=0.1):
    """Illustrative per-round logical-error fit for surface-code scaling."""
    return A * (p / p_th) ** ((d + 1) / 2)


def surface_qubits(d):
    """Data and syndrome qubits in one bare rotated surface-code patch."""
    return 2 * d**2 - 1

## 1. From detection to correction: stabilizer codes for error recovery

This section makes the relationship between detection and correction concrete. The check-based language and the notion of code distance are already familiar from the previous lesson; here we recall them briefly and look at how correction uses the same information more fully. This is the first entry in the lesson's running cost accounting: correction buys more than detection, and the price appears first in the code distance it demands — detecting a weight-$w$ error (one acting nontrivially on $w$ qubits) needs $d \ge w+1$, while correcting it needs $d \ge 2w+1$.

### 1.1 Recap and what changes for correction

The check-based language we use in this section is not new. The previous error detection module already worked with stabilizer checks and their syndromes when introducing the [[4,2,2]] detection code, and code distance appeared there as the smallest error weight a code fails to detect.

What changes in correction is how these familiar objects are used. The same checks now return a *syndrome* that must do more than confirm something went wrong: it has to narrow the possibilities enough for a decoder to choose a likely recovery. And the same code distance $d$ plays a stronger role — a code of distance $d$ can correct any error of weight up to $\lfloor (d-1)/2 \rfloor$, which is roughly twice as demanding as detection alone (detecting a weight-$w$ error needs $d \ge w+1$, correcting it needs $d \ge 2w+1$). These two upgrades — informative syndromes and a stricter use of distance — separate correction from the detection step of the previous lesson, and they set up the cost we examine in later sections.

### 1.2 A worked example: the 3-qubit repetition code

The simplest example of correction-as-decoding is the 3-qubit repetition code. One logical qubit, protected only against bit flips, is encoded into three physical qubits, and a single bit flip can be inferred from two parity checks, $Z_1 Z_0$ and $Z_2 Z_1$. The syndrome does more than signal that something went wrong: it indicates which qubit most likely flipped, so the logical value can be recovered by majority vote.

We use Watrous's *Foundations of Quantum Error Correction* as the reference for the encoding circuit, stabilizer picture, syndrome table, and logical error curve, rather than re-deriving them in full here. As it emphasizes, the repetition code is only a limited quantum code: it protects against bit-flips but not against phase errors, so it is best viewed as a pedagogical starting point rather than a practical code for general quantum computation.

Two ideas from this example carry forward. First, the syndrome is *informative*: it supports decoding, not just filtering. This is the step from detection to correction described in 1.1. Second, the logical error rate improves only when the physical error rate is below a threshold-like crossover. For the 3-bit repetition code, the decoded error rate $3p^2 - 2p^3$ is lower than the unencoded error rate $p$ only when $p < 1/2$. This simple picture foreshadows the more important threshold behavior that appears again in later sections, especially for the surface code in Part 2.

![Encoding circuit for the 3-qubit repetition code](/learning/images/courses/foundations-of-quantum-error-correction/correcting-quantum-errors/repetition-encoder.svg)

*Encoding circuit for the 3-qubit repetition code. Source: Watrous, [Foundations of Quantum Error Correction — Repetition codes](/learning/courses/foundations-of-quantum-error-correction/correcting-quantum-errors/repetition-codes), IBM Quantum Learning.*

In [2]:
# the 3-qubit repetition code


def syndrome(flip):
    qc = QuantumCircuit(5)  # q0,q1,q2 = data ; q3,q4 = syndrome ancillas

    if flip is not None:
        qc.x(flip)  # inject one bit-flip error
    qc.cx(0, 3)
    qc.cx(1, 3)  # ancilla q3 <- parity of (q0, q1) = Z1 Z0 check
    qc.cx(1, 4)
    qc.cx(2, 4)  # ancilla q4 <- parity of (q1, q2) = Z2 Z1 check
    sv = Statevector(qc)
    idx = int(np.argmax(np.abs(sv.data)))  # the state is a single basis ket
    bit = [(idx >> q) & 1 for q in range(5)]  # qiskit little-endian readout
    return (bit[3], bit[4])


# Build the syndrome table: each single-qubit flip gives a DISTINCT syndrome.
table = {}
print("error    (Z1Z0, Z2Z1)")
for flip in [None, 0, 1, 2]:
    s = syndrome(flip)
    table[s] = flip
    print(f"  {('none' if flip is None else 'flip q'+str(flip)):8s} {s}")


# The decoder is just the inverse lookup: syndrome -> most-likely flip -> undo it.
def decode(s):
    return table[s]

error    (Z1Z0, Z2Z1)
  none     (0, 0)
  flip q0  (1, 0)
  flip q1  (1, 1)
  flip q2  (0, 1)


As the table shows, each single-qubit bit-flip produces a distinct syndrome, so the syndrome alone tells the decoder which qubit to flip back.

In [3]:
# 3-bit repetition: encoded vs unencoded error.
p = np.linspace(0, 1, 400)
encoded = 3 * p**2 - 2 * p**3

fig, ax = plt.subplots(figsize=(6, 4.2))
ax.plot(
    p,
    p,
    color=plt.rcParams["grid.color"],
    linestyle="--",
    linewidth=1.4,
    label=r"unencoded: $p$",
)
ax.plot(
    p,
    encoded,
    color=colors[0],
    linewidth=2.4,
    label=r"3-bit repetition (decoded): $3p^2 - 2p^3$",
)

ax.plot(0.5, 0.5, "o", color=plt.rcParams["text.color"], markersize=5, zorder=5)
ax.annotate(
    r"crossover at $p = 1/2$",
    xy=(0.5, 0.5),
    xytext=(0.56, 0.36),
    color=plt.rcParams["text.color"],
    arrowprops=dict(arrowstyle="-", color=plt.rcParams["text.color"], lw=0.7),
)

ax.set_xlabel(r"physical flip rate $p$")
ax.set_ylabel(r"error rate after decoding")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

<Image src="/learning/images/courses/tools-for-quantum-advantage/error-correction/extracted-outputs/36475b64-0.avif" alt="Output of the previous code cell" />

The encoded error rate of the 3-bit repetition code is $3p^2 - 2p^3$ — majority vote fails when at least two of the three qubits flip, so the failure probability is $\binom{3}{2}p^2(1-p) + \binom{3}{3}p^3 = 3p^2 - 2p^3$. Plotting it against the unencoded rate $p$ makes the crossover at $p = 1/2$ explicit. The same visual logic — *encoded error below unencoded error below a crossover* — reappears for the surface code in section 2, where the crossover becomes the code threshold and the physical error rate of interest sits far below it.

This plot also explains when correction is worth its cost. For shallow circuits the advantage is small, and the lighter-weight methods from earlier lessons — suppression, mitigation, and detection-and-discard — are often enough to keep results meaningful. It is at depth that the situation changes: at larger circuit volumes, rigorous mitigation methods can require rapidly growing — often exponential — sampling overhead, and post-selected detection retains a shrinking fraction of runs. Deep, general-purpose fault-tolerant algorithms — Shor's factoring, phase estimation, the deep chemistry routines built on it — sit firmly in this regime. Correction becomes important there because it pays a large architectural overhead in physical qubits, decoding, and connectivity in exchange for an error rate that scales gently with depth. In the framework of [Lanes et al., 2025](https://arxiv.org/abs/2506.20658), near-term advantage claims may still rely on mitigation or post-selected detection where the depth and validation requirements allow; decoding-based correction becomes central when those methods cannot provide rigorous error control at the required depth, while mitigation and post-selection can continue to complement it.

#### Check your understanding

Is a syndrome a quantum state or a classical bit string, and what does it tell you?

<Accordion>
<AccordionItem title="Answer">

A classical bit string — the list of $\pm 1$ check outcomes. It carries partial information about the *error* (not about the encoded logical state, and without collapsing it); a decoder then infers the most likely error from it.

</AccordionItem>
</Accordion>

#### Check your understanding

Given a stabilizer code $[[n, k, d]]$, how do you tell whether it is used for error *detection* or error *correction*? Work out the detectable and correctable error weights for $[[4,2,2]]$, the Steane code $[[7,1,3]]$ (a 7-qubit CSS code holding one logical qubit at distance 3), and the gross code $[[144,12,12]]$.

<Accordion>
<AccordionItem title="Answer">

The same code can do either — which one is a choice of how you process the syndrome, not a property baked into the code. Detection works for any error of weight $\le d-1$; correction for weight $\le \lfloor (d-1)/2 \rfloor$ (from $d \ge w+1$ to detect, $d \ge 2w+1$ to correct a weight-$w$ error).

| code | $d$ | detect up to | correct up to |
|---|---|---|---|
| $[[4,2,2]]$ (Lesson 4) | 2 | 1 | 0 |
| Steane $[[7,1,3]]$ | 3 | 2 | 1 |
| gross $[[144,12,12]]$ | 12 | 11 | 5 |

This is exactly why $[[4,2,2]]$ was detection-only in Lesson 4: $d=2$ gives a correction radius of 0. (Two common slips: the *parity* of $d$ is irrelevant, and correction already works at $d=3$ — not only at $d \ge 4$.)

The bounds here ($d \ge w+1$ to detect, $d \ge 2w+1$ to correct) are the code's **combinatorial distance**, assuming syndrome measurement is itself noiseless. In a real fault-tolerant setting the syndrome-extraction circuit is also faulty, so recovery is governed not by $d$ alone but by the circuit-level distance and threshold under repeated noisy measurement — the realistic model used from section 3 onward.

</AccordionItem>
</Accordion>

## 2. Two-dimensional stabilizer codes and the locality-rate-distance tradeoff

Two choices frame this part's trade-off: the surface code uses *local* checks, which makes the hardware easy but lets the overhead grow, while qLDPC codes use *sparse non-local* checks, which can shrink the overhead but make the hardware and decoder harder — the rest of this part is about *why* that trade exists. The surface code is among the most studied quantum error-correcting codes, and it is a natural starting point for any discussion of correction overhead. Two features make it the reference baseline in the superconducting setting: its stabilizer checks act between nearest neighbors on a two-dimensional grid, matching the connectivity of leading hardware; and it has a comparatively high noise threshold, so a useful logical error rate can be reached at physical error rates that are realistic in practice. This section introduces the surface code at the level needed to discuss its cost, then asks where that cost comes from structurally, and finally surveys the broader landscape of codes that relax the strict 2D-locality assumption. The reason this cost is hard to avoid is structural: for any code whose checks act only between nearest neighbors on a 2D grid, the dimensionless ratio $kd^2/n$ is capped at a constant — the Bravyi–Poulin–Terhal (BPT) ceiling — and that ceiling is what holds the advantage back, as Section 2.3 develops.

### 2.1 Stabilizers on a 2D lattice

The surface code composes stabilizer checks into a two-dimensional grid. Data qubits sit on the edges of a square lattice, with ancilla qubits on the faces and at the vertices. Each ancilla measures a *weight-4 stabilizer* — either an X-type around a vertex or a Z-type around a face — on the four data qubits adjacent to it, using only nearest-neighbor two-qubit gates. The full syndrome-extraction circuit fits on a flat layout, which is the property that aligns the surface code with planar superconducting hardware.

![Weight-4 stabilizer generators of the toric/surface code: X-type around a vertex, Z-type around a face](/learning/images/courses/foundations-of-quantum-error-correction/quantum-code-constructions/toric-code-stabilizer-generators.svg)

*Two stabilizer generator types of the toric/surface code, each of weight 4. Source: [Foundations of Quantum Error Correction — Toric code](/learning/courses/foundations-of-quantum-error-correction/quantum-code-constructions/toric-code), IBM Quantum Learning.*

Two further properties make the surface code the reference toward later code families. The first is its noise *threshold*: under realistic circuit-level noise, encoding helps once physical error rates fall below roughly one percent ([Fowler et al., 2012](https://arxiv.org/abs/1208.0928)) — a level often quoted near the one-percent scale, which is one reason the surface code has become the baseline for superconducting architectures. The second is the maturity of its *decoding*: under standard Pauli-noise assumptions, inferring the most likely error from a syndrome is often formulated as a *minimum-weight matching* problem on syndrome endpoints on the lattice, a problem solved by efficient classical algorithms.

![Minimum-weight matching corrections on the toric code](/learning/images/courses/foundations-of-quantum-error-correction/quantum-code-constructions/X-error-correction.svg)

*A pair of syndrome endpoints (gray tiles) connected by a shortest-path correction (yellow). Decoding the surface code reduces to minimum-weight matching of such endpoints. Source: same lesson.*

What the figure adds is the part the syndrome leaves out. Because only the endpoints are visible, the decoder cannot see the actual error chain (magenta). It can only choose a chain with the same endpoints; minimum-weight matching picks the shortest one (yellow) in this example.

The lattice geometry also changes what a logical operator looks like. In the $[[4,2,2]]$ code from the previous lesson, logical operators were short Pauli strings of weight equal to the distance. Here they are *string operators* — chains of single-qubit Paulis along a path in the lattice. A short chain of errors creates a string with two endpoints, and the stabilizers at those endpoints fire: the syndrome reveals *where* the chain ends, even though the chain itself is not directly observed. Seeing only the endpoints, the decoder just takes the shortest chain joining them (yellow) — not necessarily the real one. That is fine: the actual errors and the correction together close a loop, and a loop that shrinks to a point is a product of stabilizers, so the qubit is recovered even when the guessed path is "wrong." A logical error, by contrast, is a string with no endpoints; in the toric code it wraps around the torus, while in the open-boundary surface code we focus on it stretches from one boundary of the patch to the other. The distance $d$ is the length of the shortest such non-trivial string, and the number of physical qubits in a distance-$d$ patch scales as $d^2$.

### 2.2 Why 2D locality creates overhead

Being below threshold is a necessary condition, not a sufficient one. To reach a *target* logical error rate — say $10^{-12}$ per round, a demanding stress-test target for deep fault-tolerant algorithms — the code distance has to grow until the residual logical-error probability drops to that level. The standard surface-code scaling has the per-round logical error behaving as

$$\varepsilon_L(d, p) \;\approx\; A\left(\frac{p}{p_{\text{th}}}\right)^{(d+1)/2},$$

with threshold $p_{\text{th}} \sim 10^{-2}$ and a code-dependent prefactor $A$ of order $0.1$. The exponent $(d+1)/2$ says that each two extra units of distance gain roughly a factor $p / p_{\text{th}}$ in logical error, so well below threshold the suppression is exponential in $d$ — but only in $d$, not in qubit count ([Fowler and Gidney, 2018](https://arxiv.org/abs/1808.06709)).

The qubit count grows quadratically: a bare distance-$d$ rotated patch uses $d^2$ data qubits and $d^2-1$ syndrome qubits, totaling $2d^2-1$ physical qubits per logical qubit ([Bravyi et al., 2024](https://www.nature.com/articles/s41586-024-07107-7)). Combining the two scalings, reaching a deep target error rate forces $d$ to grow logarithmically with the target while the physical-qubit footprint grows as the square of that. The plot below makes the trade explicit: at a physical error rate of $p = 10^{-3}$, a reasonable level for current devices, reaching an illustrative per-round target of $\varepsilon_L = 10^{-12}$ for one memory patch requires distance $d \approx 21$ and 881 physical qubits in this model. This is a per-patch memory figure, not an algorithm budget. A computation uses many patches over many rounds, plus space for routing and magic-state production, so it does not mean that 881 qubits can run $10^{12}$ logical operations. In a separate memory benchmark, [Bravyi et al., 2024](https://www.nature.com/articles/s41586-024-07107-7) estimate nearly 3,000 physical qubits in total to protect 12 logical qubits with the surface code, compared with 288 for the gross code at $p = 10^{-3}$. That comparison uses the paper's simulated memory performance, not the $10^{-12}$ target used here. Stretching the target by another factor of $10^{-2}$ in logical error rate adds only a few more units of distance, but the qubit count climbs by another few hundred. This is the overhead that motivates the search for codes outside the strict 2D-local family, which is the topic of section 3.

In [4]:
# Surface-code overhead at p = 1e-3
ds = np.arange(3, 28, 2)
qub = [surface_qubits(d) for d in ds]
ler = [surface_logical_error(d) for d in ds]
d_tera = next(d for d in ds if surface_logical_error(d) <= 1e-12 + 1e-24)

fig, ax = plt.subplots(figsize=(6, 4.2))
ax.semilogy(qub, ler, "o-", color=colors[0], linewidth=2.2, markersize=5)
ax.axhline(1e-12, color=plt.rcParams["grid.color"], linestyle="--", linewidth=1.2)
ax.text(
    qub[1],
    1.5e-12,
    r"illustrative per-round target",
    color=plt.rcParams["axes.edgecolor"],
)
ax.scatter(
    [surface_qubits(d_tera)],
    [surface_logical_error(d_tera)],
    s=90,
    color=plt.rcParams["text.color"],
    zorder=5,
)
ax.annotate(
    rf"$d = {d_tera}$, {surface_qubits(d_tera)} qubits/logical",
    xy=(surface_qubits(d_tera), surface_logical_error(d_tera)),
    xytext=(surface_qubits(d_tera) + 90, surface_logical_error(d_tera) * 5),
    color=plt.rcParams["text.color"],
    arrowprops=dict(arrowstyle="-", color=plt.rcParams["text.color"], lw=0.7),
)

ax.set_xlabel(r"physical qubits per logical qubit  $\;[2d^2-1]$")
ax.set_ylabel(r"logical error per round  $\varepsilon_L$")
ax.set_title(
    "Schematic toy fit (A=0.1, p_th=1e-2) — illustrative shape, not a resource estimate",
    fontsize=9,
)
plt.tight_layout()
plt.show()

<Image src="/learning/images/courses/tools-for-quantum-advantage/error-correction/extracted-outputs/5b45bf47-0.avif" alt="Output of the previous code cell" />

The marked point shows the basic overhead message. In this fit, reaching the illustrative per-round target at $p = 10^{-3}$ requires distance $d \approx 21$, or about $881$ physical qubits per logical qubit using the bare rotated-patch count $2d^2-1$. This is not a full architecture-level resource estimate: a real fault-tolerant system must also budget syndrome-extraction circuits, decoding latency, routing, logical operations, and non-Clifford resources (the universal-gate machinery of section 4). The point is the scaling pressure that motivates looking beyond strictly 2D-local codes, which is the topic of the rest of this section.

*(These figures trace a toy-fit curve, not a resource estimate)*

### 2.3 Locality, rate, and distance: the Bravyi–Poulin–Terhal bound

A useful way to compare codes is by how many logical qubits they pack into a given number of physical qubits — the *rate*, written $k/n$, where $k$ is the number of logical qubits encoded and $n$ is the number of physical qubits used. The surface code has rate $k/n = 1/d^2$: each logical qubit occupies its own $d \times d$ patch, and the rate vanishes as the distance grows.

This vanishing rate is a property of the geometry, not of the surface code in particular. A result of Bravyi, Poulin, and Terhal ([Bravyi et al., 2010](https://arxiv.org/abs/0909.5200)) — the *BPT bound* — says that for any stabilizer code whose checks act only between nearest neighbors on a 2D grid, the product $k\,d^2$ cannot exceed the number of physical qubits up to a constant. Equivalently, the dimensionless ratio $k\,d^2/n$ is capped by a constant for every code in this family. The surface code in its various forms — rotated, planar, toric — all sit near the same $O(1)$ value, with the exact constant depending on how the boundary is handled.

The consequence is a hard tradeoff. Within strict 2D locality, increasing the distance to suppress the logical error rate forces the rate to shrink, and the physical-qubit count per logical qubit grows quadratically with distance. Within strict 2D locality, rate and distance cannot both be large — their product $k\,d^2$ is capped at $O(n)$ (the BPT bound). Codes that leave the 2D-local family can do better, by an order of magnitude or more. In the table below, $n$ is the code block length used in the code parameters $[[n,k,d]]$; it is not a complete hardware footprint, since syndrome ancillas, routing, couplers, and measurement hardware are not included.

| code | family | $[[n, k, d]]$ | $k\,d^2/n$ | reference |
|---|---|---|---|---|
| rotated surface | 2D-local (surface) | $[[d^2,\,1,\,d]]$ | $1$ | [Bombín and Martín-Delgado, 2007](https://arxiv.org/abs/quant-ph/0703272) |
| planar surface | 2D-local (surface) | $[[2d^2 - 2d + 1,\,1,\,d]]$ | $\to 1/2$ | [Bravyi and Kitaev, 1998](https://arxiv.org/abs/quant-ph/9811052) |
| toric | 2D-local (surface) | $[[2d^2,\,2,\,d]]$ | $1$ | [Kitaev, 2003](https://arxiv.org/abs/quant-ph/9707021) |
| BB | bivariate bicycle | $[[72,\,12,\,6]]$ | $6$ | [Bravyi et al., 2024](https://www.nature.com/articles/s41586-024-07107-7) |
| gross | bivariate bicycle | $[[144,\,12,\,12]]$ | $12$ | [Bravyi et al., 2024](https://www.nature.com/articles/s41586-024-07107-7) |
| two-gross | bivariate bicycle | $[[288,\,12,\,18]]$ | $13.5$ | [Bravyi et al., 2024](https://www.nature.com/articles/s41586-024-07107-7) |

The three codes in the lower half of the table — the *bivariate-bicycle* family, including the *gross* and *two-gross* codes — are the case study of section 3. Their sparse checks are geometrically non-local relative to a strict 2D layout, and that is what lets them move above the surface-code locality ceiling. What those non-local checks cost in connectivity, decoding, and architecture is the rest of the story this section sets up.

<Admonition type="note" title="How to read from here on">

Many names appear from here on — BB / gross code, qLDPC, Tanner graph, Kookaburra, Cockatoo, and Starling. You do not need to memorize or fully understand them. Track only three moving costs, and ask of each new name "which cost does this touch?":

1. **code structure** — how the checks are defined (Part 3)
2. **decoder cost** — the classical compute that processes syndromes (Section 5.1–5.2)
3. **hardware connectivity** — the wiring the checks demand (Section 5.3)

The recurring story is that qLDPC lowers cost (1)'s qubit count by *raising* costs (2) and (3).

</Admonition>

### 2.4 The qLDPC landscape

A *quantum LDPC* (qLDPC) code is a stabilizer code whose *parity-check matrix* — the binary array whose rows are the stabilizer checks (one row per check, one column per qubit, a 1 wherever a check acts on a qubit) — is *sparse*: each row and column has only a small, bounded number of 1s, regardless of code size — that is what "low-density" means. By this definition the surface code is itself a qLDPC code; its weight-4 stabilizers do not grow with $n$. What sets the qLDPC codes of recent interest apart is not that they are sparser than the surface code, but that they relax 2D-locality while keeping sparsity. That relaxation is what allows them to escape the BPT ceiling examined in Section 2.3.

The "LDPC" terminology comes from classical coding theory, where low-density parity-check codes have long been known to combine good rates with efficient decoding. The quantum analogue inherits the structural advantage of sparse checks while removing the geometric-locality assumption. Several families realize this: *hypergraph-product* codes ([Tillich and Zémor, 2009](https://arxiv.org/abs/0903.0566)), *lifted-product* codes ([Panteleev and Kalachev, 2022](https://arxiv.org/abs/2111.03654)), *quantum Tanner* codes ([Leverrier and Zémor, 2022](https://arxiv.org/abs/2202.13641)), and *bivariate-bicycle* codes ([Bravyi et al., 2024](https://www.nature.com/articles/s41586-024-07107-7)). Some of these are *asymptotically good* — keeping constant rate and linear distance as the code grows — a property surface codes cannot achieve.

For this lesson the concrete case study is the bivariate-bicycle family, examined in detail in Part 3. The reason for that choice is not a mathematical claim that bivariate-bicycle is the strongest qLDPC family — it is an architectural alignment: IBM's published superconducting roadmap selects these codes because their thickness-2 check graph fits the proposed superconducting hardware, within the modular architecture discussed in Part 5. Within that scope, the headline result is that, *in the reported circuit-level memory simulation*, bivariate-bicycle codes can match surface-code suppression with about an order of magnitude fewer physical qubits per logical qubit at $p = 10^{-3}$ ([Bravyi et al., 2024](https://www.nature.com/articles/s41586-024-07107-7); see also IBM's [fault-tolerance roadmap](https://www.ibm.com/quantum/blog/large-scale-ftqc)). The cost does not disappear: it moves from physical-qubit count into *connectivity* (the check graph splits into two planar subgraphs and includes long-range edges), *decoding* (minimum-weight matching no longer applies, and qLDPC decoding is an active research area), and *architecture* (modular hardware to support both). Part 3 turns this qLDPC idea into a concrete BB / gross-code case study; later sections return to logical operations, decoding, and architecture.

![Gross code visualized on a 3D torus](/learning/images/courses/tools-for-quantum-advantage/error-correction/gross-code-torus.gif)

*Visualization of the gross code $[[144,\,12,\,12]]$: 12 logical qubits encoded across 144 data qubits, with 144 syndrome qubits. Realizing the code on a planar quantum chip requires long-range connections that follow the symmetry of a 3D torus — these are the non-local checks that escape the BPT bound. Source: [fault-tolerance roadmap](https://www.ibm.com/quantum/blog/large-scale-ftqc).*

#### Check your understanding

Why does the surface code map so naturally onto 2D nearest-neighbor hardware?

<Accordion>
<AccordionItem title="Answer">

Its checks are *local* — each acts on only a few neighboring qubits, so it lays directly onto a planar grid with nearest-neighbor couplers. That convenience is exactly what the BPT ceiling charges for in rate, and what qLDPC codes trade away.

</AccordionItem>
</Accordion>

#### Check your understanding

BB / gross codes can match surface-code suppression with about $10\times$ fewer physical qubits — for the memory task, in the reported circuit-level simulation. Where does the "saved" cost reappear?

<Accordion>
<AccordionItem title="Answer">

It does not vanish; it *moves*. The weight-6 non-local checks demand long-range connectivity, implemented with c-couplers within a chip in IBM's proposal. Scaling the modular architecture also calls for inter-module l-couplers (Section 5.3). Decoding uses methods such as BP+OSD or Relay-BP rather than surface-code matching (Section 5.2). (It is *not* that qLDPC is strictly better — Section 3.5 — and the noise threshold and syndrome depth stay comparable to the surface code.)

</AccordionItem>
</Accordion>

## 3. BB / gross codes as a qLDPC case study

Section 2 placed the bivariate-bicycle codes in the qLDPC landscape and showed why, in the reported memory-task simulations, selected qLDPC codes can match a surface-code memory target at much lower physical-qubit overhead. This section turns to the BB family as a sustained case study. We first work through how the construction actually produces a code (Section 3.1), then examine the memory performance that placed the gross code on IBM's roadmap (3.2), how the family scales with distance (3.3), what computing on a BB code adds beyond memory (3.4), and what the case study collectively does and does not establish (3.5). Decoding and modular hardware return in Part 5 as parts of the broader system view; the universal-gate machinery sits in Part 4. Allowing non-local checks steps *outside* BPT's 2D-local assumption — without violating the bound — and the payoff, for the memory task this part develops, is matching surface-code suppression with about an order of magnitude fewer physical qubits.

### 3.1 The bivariate-bicycle construction

The bivariate-bicycle (BB) construction generates an entire stabilizer code from two short polynomials $A$ and $B$. Where the surface code is built by drawing local plaquettes on a 2D grid, BB generates its parity-check matrix *algebraically*: a single rule produces all 72 X-checks and all 72 Z-checks of the gross code from the polynomial pair $(A, B)$. The construction itself guarantees that the resulting X- and Z-checks commute — i.e., that the result is a valid CSS code — with no commutation conditions to enforce by hand. Whether a given $(A,B)$ yields useful rate, distance, and decoding properties is then checked numerically — which is what lets the design be a *polynomial search* rather than a geometric construction.

For the gross code, the polynomials are $A = x^3 + y + y^2$ and $B = y^3 + x + x^2$ over a $12 \times 6$ torus lattice with 144 data qubits — here $x$ and $y$ are the lattice's two cyclic shift operators ("move one step" along each torus direction), not numbers, so a term like $x^3$ means "shift three steps" (made precise in the foldable below). The 144 data qubits split into two registers, $q(L)$ and $q(R)$, each containing 72 qubits. Each weight-6 check touches three qubits in each register. The *biplanar* or *thickness-2* property means that the Tanner graph splits into two edge-disjoint planar subgraphs, allowing couplers to be routed in two layers. It does not place the two data registers on separate physical planes ([Bravyi et al., 2024](https://www.nature.com/articles/s41586-024-07107-7)). The figure below renders the resulting check matrix alongside a surface-like code on the same lattice, showing how different polynomial choices within the same framework produce very different parity-check patterns. The figure shows which data qubits each check touches — it is not a hardware-placement diagram.

In [5]:
# Surface-like local checks vs Gross-code BB checks

from matplotlib.colors import ListedColormap

ell, m = 12, 6
S_l = np.roll(np.eye(ell, dtype=int), 1, axis=1)
S_m = np.roll(np.eye(m, dtype=int), 1, axis=1)
I_l, I_m, I_lm = (
    np.eye(ell, dtype=int),
    np.eye(m, dtype=int),
    np.eye(ell * m, dtype=int),
)
x = np.kron(S_l, I_m)
y = np.kron(I_l, S_m)


def mpow(M, p):
    out = np.eye(M.shape[0], dtype=int)
    for _ in range(p):
        out = (out @ M) % 2
    return out


# Toric (surface-like): A = 1 + x, B = 1 + y  →  weight 4 per row
H_toric = np.concatenate([(I_lm + x) % 2, (I_lm + y) % 2], axis=1)
# Gross: A = x^3 + y + y^2, B = y^3 + x + x^2  →  weight 6 per row
H_gross = np.concatenate(
    [
        (mpow(x, 3) + mpow(y, 1) + mpow(y, 2)) % 2,
        (mpow(y, 3) + mpow(x, 1) + mpow(x, 2)) % 2,
    ],
    axis=1,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
cmap_blue = ListedColormap([plt.rcParams["axes.facecolor"], colors[0]])
cmap_accent = ListedColormap([plt.rcParams["axes.facecolor"], colors[1]])

panels = [
    (
        axes[0],
        H_toric,
        cmap_blue,
        "Toric code $[[144, 2, 6]]$",
        "$A = 1 + x$,   $B = 1 + y$",
        "weight 4 per row,  ~2.8% density",
    ),
    (
        axes[1],
        H_gross,
        cmap_accent,
        "Gross code $[[144, 12, 12]]$",
        "$A = x^{3} + y + y^{2}$,   $B = y^{3} + x + x^{2}$",
        "weight 6 per row,  ~4.2% density",
    ),
]
for ax, H, cmap, title, poly, density in panels:
    ax.imshow(H, cmap=cmap, aspect="auto", interpolation="nearest")
    ax.axvline(
        72 - 0.5, color=plt.rcParams["grid.color"], linewidth=0.8, linestyle="--"
    )
    ax.set_xlabel("data qubit index")
    ax.set_ylabel("X-check index")
    ax.set_title(title, pad=8)
    ax.text(
        0.5,
        -0.18,
        poly,
        transform=ax.transAxes,
        color=plt.rcParams["axes.edgecolor"],
        ha="center",
    )
    ax.text(
        0.5,
        -0.27,
        density,
        transform=ax.transAxes,
        color=plt.rcParams["axes.edgecolor"],
        ha="center",
    )

plt.tight_layout()
plt.show()

# Cheap sanity check: the construction yields a valid CSS code (H_X H_Z^T = 0 mod 2).
A_g = (mpow(x, 3) + mpow(y, 1) + mpow(y, 2)) % 2
B_g = (mpow(y, 3) + mpow(x, 1) + mpow(x, 2)) % 2
H_X = np.concatenate([A_g, B_g], axis=1)  # X-checks  [A | B]
H_Z = np.concatenate([B_g.T, A_g.T], axis=1)  # Z-checks  [B^T | A^T]


assert not np.any((H_X @ H_Z.T) % 2)

<Image src="/learning/images/courses/tools-for-quantum-advantage/error-correction/extracted-outputs/46c4cfdf-0.avif" alt="Output of the previous code cell" />

The two matrices live in the same $72 \times 144$ grid but with very different patterns. The toric code's weight-4 rows pack into tight diagonal stripes — each check connects only to its four neighboring data qubits. The gross code's weight-6 rows spread across more diagonals at wider spacing because the $x^3$ and $y^3$ terms reach three steps across the torus. The polynomial choice *is* the code, and at this level the difference between a 2D-local code and a non-local qLDPC code is just the choice of $A$ and $B$.

The construction comes with three properties worth naming: it always produces a valid CSS code (X- and Z-checks commute algebraically — verified directly in the cell above); the block length $n$ and number of logical qubits $k$ follow from $\mathbb{F}_2$-rank computations on the polynomial pair ($k = n - \mathrm{rank}\,H_X - \mathrm{rank}\,H_Z$), with the gross parameters $[[144,12,12]]$ taken from [Bravyi et al., 2024](https://www.nature.com/articles/s41586-024-07107-7) — the distance in particular is quoted, not recomputed, since finding the distance of a general qLDPC code is hard; and the sparse but non-local checks still detect errors effectively. The detailed proofs and the detection argument are in the foldable below for readers who want them; they are not required to follow the rest of section 3.

Read together, the two panels make a point that sparsity alone hides. The toric matrix *looks* leaner — weight-4 rows, tighter stripes, ~2.8% density — yet at the same $n = 144$ it is only a $[[144, 2, 6]]$ code: 2 logical qubits, distance 6, correcting weight-2 errors. The denser-looking gross matrix is $[[144, 12, 12]]$: six times the logical qubits and twice the distance (correcting weight-5 errors) on the *same* 144 data qubits. So the extra check weight is not waste — the non-local weight-6 checks are exactly what buy the higher rate and distance, i.e. far lower overhead *per logical qubit*. This is the BPT ceiling of Section 2.3 in concrete form: the toric layout stays at order $O(1)$, while the gross code reaches $k\,d^2/n = 12$.

The cost of these non-local checks reappears at decoding time. Because the checks are non-local, the syndrome graph a decoder must work on is no longer the planar matching graph of the surface code, and minimum-weight matching does not apply directly. The structure of that decoding problem, and the techniques developed for it, are in Section 5.2.

<Accordion>
<AccordionItem title="BB construction proofs: CSS commutation, $[[144,12,12]]$ parameters, detection of non-local errors (deeper dive)">

**The compressed wiring rule.**

Take a $12 \times 6$ periodic lattice — a torus with 72 positions $(i, j)$ where $0 \le i < 12$ and $0 \le j < 6$. Define two cyclic shift operators on this lattice: $x$ moves each position one step horizontally — $(i, j) \mapsto (i+1 \bmod 12,\ j)$ — and $y$ moves one step vertically. Read $x$ and $y$ as *operators*, not as numbers: $x$ is the action "move one step right", $y$ is "move one step up". Because the lattice is periodic, $x^{12} = 1$ and $y^6 = 1$. Because the shifts commute, $xy = yx$. The BB construction takes two polynomials $A, B$ in $x, y$ — elements of the ring $\mathbb{F}_2[x, y]\,/\,(x^{12} - 1,\, y^6 - 1)$ — and builds the parity-check matrices as $H_X = [\,A \mid B\,]$ and $H_Z = [\,B^\top \mid A^\top\,]$.

For the gross code, $A = x^3 + y + y^2$ and $B = y^3 + x + x^2$. Reading $A$: each X-check at lattice position $(i, j)$ touches three $q(L)$ qubits at positions $(i+3,\,j)$, $(i,\,j+1)$, $(i,\,j+2)$, modulo the lattice dimensions. $B$ adds three further $q(R)$ qubits at $(i,\,j+3)$, $(i+1,\,j)$, $(i+2,\,j)$. Same rule at every lattice position gives 72 X-checks, 72 Z-checks, total $n = 2 \cdot 12 \cdot 6 = 144$ data qubits.

**Why the construction always gives a CSS code.** A CSS code requires X- and Z-checks to commute as Pauli operators. Writing each Pauli on the $n$ qubits as a pair of binary vectors — one marking its $X$-support, one its $Z$-support — two Paulis commute exactly when those supports overlap in an *even* number of positions (their symplectic inner product is $0 \bmod 2$); since an X-check carries only $X$'s and a Z-check only $Z$'s, that condition for the whole code collapses to the matrix equation $H_X H_Z^\top \equiv 0 \pmod 2$. Compute

$$H_X H_Z^\top = [\,A \mid B\,] \begin{bmatrix} B \\ A \end{bmatrix} = AB + BA \pmod 2.$$

Because $A$ and $B$ are polynomials in commuting operators $x, y$, they commute as operators: $AB = BA$. So $AB + BA = 2AB \equiv 0 \pmod 2$. Any pair of polynomials over $\mathbb{F}_2[x, y]\,/\,(x^l - 1,\, y^m - 1)$ automatically gives a CSS pair, no commutation condition to enforce by hand.

**Where $[[144, 12, 12]]$ comes from.** The block length $n = 2lm$ is the lattice geometry. The number of logical qubits is $k = n - \mathrm{rank}(H_X) - \mathrm{rank}(H_Z)$; for the gross code's $A$ and $B$ it equals 12. The distance $d$ is the minimum weight of a non-trivial logical operator, computed numerically; for the gross code, $d = 12$. The polynomials $A, B$ were found by numerical search with $l = 12$, $m = 6$, weight-6 checks, and a thickness-2 Tanner graph, maximizing rate and distance ([Bravyi et al., 2024](https://www.nature.com/articles/s41586-024-07107-7)).

**Why sparse non-local checks still detect errors.** A Pauli error is detected when it anticommutes with at least one stabilizer check. Errors that commute with every check are either products of stabilizers (harmless) or non-trivial logical operators (corrupting). Since the code distance $d$ is the minimum weight of a non-trivial logical operator, any Pauli error of weight strictly less than $d$ is either harmless or anticommutes with some check. For the gross code, $d = 12$, so any Pauli error on fewer than 12 data qubits is either detected or acts trivially on the codespace.

A subtlety on what a single check sees: an X-type check anticommutes with $Z$ and $Y$ on its support but is blind to $X$ errors there (and Z-type checks are symmetric). So even a check whose support contains an error may not fire — two $Z$ errors on the same X-check's support each anticommute with it, but together commute. The distance guarantee — that *some* check fires for any Pauli error of weight below $d$ that acts nontrivially on the codespace — is a property of the *full* check network, not of any single check. Each data qubit participates in three X-checks and three Z-checks, and those supports overlap in patterns determined by the polynomials. The algebraic structure ensures coverage without gaps; the non-local edges make the design tight enough to give $d = 12$ on only 144 data qubits — a distance that would require ~$d^2 = 144$ qubits *per logical qubit* in a 2D-local rotated surface patch.

</AccordionItem>
</Accordion>

### 3.2 Memory: Bravyi et al. 2024 — the gross code $[[144, 12, 12]]$

Section 2.2 showed that surface-code memory is expensive at deep logical-error targets, and storing many logical qubits compounds the cost: each logical needs its own $d \times d$ patch, and the per-logical budget grows quadratically with distance. The BPT bound (Section 2.3) explains structurally why this overhead is hard to reduce within strict 2D locality. The natural question is whether stepping outside that constraint can lower the memory budget at concretely useful distances, while keeping the threshold and decoder properties that made surface codes the standard baseline.

The paper introduces the bivariate-bicycle (BB) family, develops the construction worked through in Section 3.1, and proposes specific code instances for evaluation. The headline candidate is the *gross code* $[[144, 12, 12]]$: one block holds 12 logical qubits in 144 data qubits plus 144 syndrome ancillas, for 288 physical qubits in total. Its Tanner graph makes the structure concrete: every vertex has six incident edges — four short-range (north, south, east, west) and two long-range — with the data qubits split across two registers $q(L)$ and $q(R)$.

![Tanner graphs of the surface code (a) and the gross BB code [[144,12,12]] embedded into a torus (b), with four short-range and two long-range edges per vertex](/learning/images/courses/tools-for-quantum-advantage/error-correction/tanner-graphs.avif)

*Tanner graphs of (a) the surface code and (b) the gross BB code $[[144, 12, 12]]$ embedded into a torus. Each data/check vertex has six incident edges — four short-range plus two long-range (only a few long-range edges are drawn, to avoid clutter); blue and orange circles are the $q(L)$ and $q(R)$ data registers. Source: [Bravyi et al., 2024](https://www.nature.com/articles/s41586-024-07107-7), Nature 627, 778; figure via IBM Quantum blog, [Nature paper announcement](https://www.ibm.com/quantum/blog/nature-qldpc-error-correction).*

The full simulation evaluates a short, constant-depth syndrome-measurement cycle (seven layers of CNOT gates) on that non-local connectivity graph, under a *circuit-level* noise model — stochastic Pauli faults applied at every gate and measurement of the syndrome circuit, including the ancillas and the operations themselves, not just at idle data qubits. (This is the realistic model for hardware estimation; *code-capacity* models that fault only the data are abstract and less predictive of hardware performance.) The specific parameters here use standard superconducting-style depolarizing noise strengths (under *depolarizing* noise each qubit independently suffers a random $X$, $Y$, or $Z$ error with equal probability). The published paper compares the gross code with surface-code memories storing the same 12 logical qubits in independent rotated patches. Each patch uses $2d^2-1$ physical qubits, including data and syndrome qubits; the distance-11 reference therefore uses 241 qubits per patch.

Under that circuit-level model, the gross code preserves all 12 logical qubits for on the order of a million error-correction cycles at $p = 10^{-3}$, using 288 physical qubits in total ([Bravyi et al., 2024](https://www.nature.com/articles/s41586-024-07107-7); IBM's [fault-tolerance roadmap](https://www.ibm.com/quantum/blog/large-scale-ftqc) blog). The paper also reports a *threshold* of about 0.7% under the same circuit-level noise model, comparable to the surface-code threshold and a precondition for hardware viability. (A *threshold* is the asymptotic physical-error rate below which increasing code distance reduces logical error; a *pseudo-threshold* is the empirical crossover measured for specific finite distances under a specific noise model. The two are typically close but not identical.) The paper reports roughly an order-of-magnitude reduction in physical-qubit count for the memory benchmark; the compared codes have different logical-error curves. The same paper presents a small family of related BB codes — $[[72, 12, 6]]$, $[[90, 8, 10]]$, $[[108, 8, 10]]$, $[[144, 12, 12]]$, $[[288, 12, 18]]$, $[[360, 12, \le 24]]$ — characterizing how the rate, distance, and required hardware budget trade off as the polynomials and lattice dimensions are varied (Bravyi et al. 2024, Table 1 and Extended Data Table 1). The headline comparison is easiest to keep honest in a table before plotting it:

| Memory for 12 logical qubits ($p = 10^{-3}$) | Physical qubits | vs. gross | Source |
|---|---:|---:|---|
| Gross code $[[144, 12, 12]]$ | 288 (144 data + 144 ancilla) | 1× | Bravyi et al. 2024 |
| Surface code (12 patches, $d = 11$) | 2,892 ($12 \times 241$) | ~10× | Distance-11 reference in Bravyi et al. 2024 |

The table compares physical-qubit counts using the paper's distance-11 surface-code reference. It does not assert equal logical-error rates at that distance; the performance comparison is given by the paper's circuit-level simulations.

What it establishes is that for the *memory* task — preserving encoded logical states across repeated error-correction cycles, with no logical gates yet performed — qLDPC codes with weight-6 checks on a 2D-superconducting layout can match the memory target in the reported simulation at roughly an order of magnitude smaller physical-qubit footprint. The demonstration is via realistic circuit-level simulation rather than abstract code-capacity bounds, which is the basis on which surface-code overhead numbers are also reported, and the reported threshold of about 0.7% (0.8% in the earlier preprint), on par with the surface code, indicates the gain is not bought by sacrificing tolerance for physical noise.

What it does not establish is the corresponding result for *computation* on the encoded qubits, which requires logical Clifford and non-Clifford operations that the memory protocol does not exercise — those are the subject of Section 3.4. It also does not yet establish hardware feasibility: the simulations assume the seven-layer syndrome cycle on a connectivity graph that current commercial superconducting chips do not implement, motivating the chip-level architecture program — c-couplers, biplanar layout, and the Loon processor (introduced in Section 5.3) — that is needed to turn the simulated code into a buildable design. The remaining subsections of section 3 work through these missing pieces — scaling (3.3) and computation (3.4) — and decoding and modular hardware are taken up in section 5.

### 3.3 Scaling the case study: two-gross

The gross code is one entry in the BB family rather than a single endpoint. The natural next question is whether the same construction extends to higher distance without changing the broader architecture, and what the qubit cost of that extension looks like. The paper's answer is the *two-gross code* $[[288, 12, 18]]$, obtained from a related polynomial choice on a larger lattice within the same bivariate-bicycle framework. The code keeps the same 12 logical qubits, raises the distance from 12 to 18, and doubles the data-qubit count from 144 to 288 (with the total physical-qubit count rising from 288 to 576 once syndrome ancillas are included).

Reading the BPT-ratio entry from Section 2.3: gross sits at $k\,d^2 / n = 12$ and two-gross sits at $12 \cdot 18^2 / 288 = 13.5$. These ratios illustrate the benefit of relaxing strict 2D locality; the BPT bound does not apply to these non-local checks. Gross and two-gross are two finite-distance operating points: the former uses fewer qubits, while the latter spends a doubled data-qubit budget to reach a higher distance. The progression from gross to two-gross increases the qubit and connectivity requirements. IBM's modular architecture (Section 5.3) is designed to connect memory, processing, and other modules as the system scales. Section 5 returns to this as an IBM case study in hardware-code co-design.

What this scaling step establishes is that the BB family extends by *changing the polynomial choice on a larger lattice*, not by replicating independent code blocks (as a surface-code memory does when growing across multiple logical qubits). The same algebraic framework, the same syndrome-extraction circuit shape, and the same decoding problem class apply across the family. These finite-size examples do not establish *asymptotic goodness*, which requires a nonzero limiting rate and distance growing linearly with code length. The case-study point is that within the *finite-distance regime* relevant to near-term fault-tolerance roadmaps, the BB family offers concrete operating points whose costs can be evaluated together with their decoder and connectivity requirements.

### 3.4 What computation adds beyond memory

The memory result of Section 3.2 protects encoded logical states; it does not perform operations on them. A useful computation requires logical *gates* — at minimum a universal set composed of Clifford and non-Clifford operations — and a way to apply them between code blocks. The BB family carries an additional structural challenge here: because checks span the chip non-locally, even simple operations within a single BB block require care to preserve the code's properties, and operations *between* BB blocks require architectural pieces — couplers, adapters, magic-state factories — that are not part of the memory construction.

The IBM proposal taken up here as a case study is *Tour de gross* ([Yoder et al., 2025](https://arxiv.org/abs/2506.03094)), which introduces the *bicycle architecture*: a fault-tolerant *system* composed of independently fabricated modules. Each code module contains a memory code block and an attached *logical processing unit* (LPU) implementing logical operations via generalized lattice surgery. Separate magic-state factory modules supply non-Clifford resources, and *universal adapters* implement inter-module logical operations. The same paper evaluates the architecture against six design criteria — *fault-tolerant* (logical errors are suppressed sufficiently for the intended algorithms to succeed), *addressable* (individual logical qubits can be prepared or measured during the computation), *universal* (the full logical gate set — Clifford plus a non-Clifford — is available), *adaptive*, *modular* (built from independently fabricated, interchangeable modules), and *efficient* (low physical-qubit and time overhead per logical operation). In the paper's terms, *adaptive* means that syndrome measurements are decoded in real time and can change the quantum instructions that follow — i.e., genuine real-time feedforward. (Throughout this lesson we instead treat corrections as an *offline frame update*: a deliberate simulation/pedagogical convenience that lets the code cells stay short and exact. Real fault-tolerant computation requires the adaptive, real-time feedforward capability above — the $S$ correction after $T$ injection and the lattice-surgery byproducts in section 4 are exactly the points where later gates depend on earlier measurement outcomes, so they cannot all be deferred offline.) The paper reports end-to-end resource estimates suggesting that order-of-magnitude larger logical circuits can be executed at a given physical-qubit budget than under surface-code architectures.

![Modular fault-tolerant architecture for BB codes — code block, LPU, adapter, T-factory](/learning/images/courses/tools-for-quantum-advantage/error-correction/bicycle-architecture.avif)

*The bicycle architecture: a fault-tolerant modular system based on bivariate-bicycle codes. Each code module combines a memory code block with a logical processing unit (LPU) for logical operations via generalized lattice surgery. Separate magic-state factory modules supply non-Clifford resources, and universal adapters connect modules. Source: [fault-tolerance roadmap](https://www.ibm.com/quantum/blog/large-scale-ftqc); architectural details in [Yoder et al., 2025](https://arxiv.org/abs/2506.03094).*

The key question is whether the memory-level advantage from Section 3.2 survives the additional overhead of computation — logical gates, magic-state resources, inter-module routing, decoding, and hardware layout. The case-study argument in Yoder et al. 2025 is that a BB-based architecture can remain competitive with surface-code-based architectures after these overheads are included, preserving part of the memory advantage. The precise margin at the full-algorithm level is still an active resource-estimation question, and the supporting evidence is architectural analysis rather than an experimental demonstration. Part 4 of this lesson works through the universal-gate side of fault-tolerant computation — Clifford structure, non-Clifford resources, and magic-state distillation, including the *single-block distillation* proposal of [Xu et al., 2026](https://arxiv.org/abs/2602.20546), which runs a magic-state factory *inside a single BB code block* — reusing the block's own logical capacity via *qubit recycling* (repurposing the same physical qubits for successive distillation steps) rather than dedicating a separate code block.

### 3.5 What this study shows — and what it does not prove

Across Sections 3.1 through 3.4, we walked through the BB construction (3.1), the memory benchmark that placed the gross code on IBM's roadmap (3.2), the family's scaling behavior to higher distances (3.3), and what architectural pieces computation adds beyond memory (3.4). What the case study collectively *shows* is a coherent multi-paper trajectory: a fully-specified qLDPC code family, a published memory accounting at realistic circuit-level noise, a demonstrated scaling step to higher distance within the same construction, and a system-level architecture proposal that addresses universal computation on the encoded qubits.

What the study does *not* prove:

- **Optimality.** Bivariate-bicycle codes are not necessarily the optimal qLDPC choice for any specific algorithm or platform. Hypergraph-product, lifted-product, and quantum Tanner codes exist and have their own asymptotic and finite-size advantages. The BB selection in IBM's roadmap is an *architectural alignment* — their thickness-2 check graphs suit the proposed superconducting hardware and modular architecture — not a mathematical optimality claim.

- **Algorithmic resource counts.** The roughly order-of-magnitude *memory* reduction reported by Bravyi et al. is for a specific noise model and target. Real algorithms have additional resource requirements — gates, magic-state distillation, routing, real-time decoder latency — that the memory accounting does not capture. The most honest summary is *an order-of-magnitude reduction at the memory task, with computational and system-level overheads still being mapped out*.

- **Hardware existence.** The connectivity required by the gross code at full scale has not been built at the time of writing. IBM's published chip roadmap (the lineage table in Section 5.3) is the proposed path to building it. Whether the timeline holds is a matter of engineering execution rather than of any published code-theory result.

The bivariate-bicycle case study is an instance of how a recent QEC innovation works through the full stack — code, circuit, architecture, decoder, integration plan — for one specific platform. Part 4 turns to the universal fault-tolerant computation problem in its own right, going deeper into Clifford and non-Clifford structure. Section 5 returns to QEC as a quantum-classical system across decoders, modular hardware, and other platforms, where the *Relay-BP* decoder ([Müller et al., 2025](https://arxiv.org/abs/2506.01779)) and the IBM modular-chip roadmap re-enter as the system-level companions to this code-level story.

#### Check your understanding

Bravyi et al. 2024 report that the gross code $[[144,12,12]]$ holds 12 logical qubits for $\sim\!10^6$ cycles at $p=10^{-3}$ using 288 physical qubits — about $10\times$ fewer than equivalent surface-code memory. State precisely what this does, and does not, establish.

<Accordion>
<AccordionItem title="Answer">

It establishes a *memory* advantage, validated by circuit-level *simulation*. It does **not** show the advantage survives full computation (logical gates, magic-state distillation, and decoder latency all add cost — Section 3.4 onward), it is **not** an experimental hardware demonstration, and it does **not** make the surface code obsolete (still the baseline). Section 3.4 (Tour de gross) argues the advantage *may* carry into computation, but that is an architectural analysis, not yet an experiment.

</AccordionItem>
</Accordion>

## 4. Universal fault-tolerant computation

Section 3 showed that the bivariate-bicycle case study makes memory cheaper and that the bicycle architecture addresses computation at the system level. This section steps back from the BB-specific picture and works through the *universal-gate primitives* that any fault-tolerant computation needs: Clifford operations, a non-Clifford gate, and distillation. The presentation is platform-independent until Section 4.4, which returns to BB codes for one case-study-specific contribution. Memory is only one line of the cost ledger: computation moves cost into non-Clifford resources, and the real discount shows only once that is summed in — the advantage ledger is the whole system, not the single memory line.

The goal here is not to derive magic-state distillation in full, but to understand why protected logical memory is not enough for universal fault-tolerant computation.

### 4.1 Clifford structure and the need for non-Clifford resources

Before the details, the shape of this section in one table:

| Operation | Why it is easy or hard on a stabilizer code |
|---|---|
| Clifford gates ($H$, $S$, CNOT) | handled relatively naturally — they preserve Pauli structure |
| $T$ gate | provides a non-Clifford resource; implemented here through a *magic state* |
| Magic-state distillation | converts many noisy magic states into fewer cleaner ones, at a cost |

The *Clifford group* $\mathcal{C}$ is the set of unitaries that map Pauli operators to Pauli operators under conjugation; standard generators are the Hadamard gate $H$, the phase gate $S = \mathrm{diag}(1, i)$, and the two-qubit $\mathrm{CNOT}$. Clifford operations preserve Pauli structure: encoding, logical Clifford gates, and stabilizer measurement can all be described within this framework. For the surface code, logical Cliffords are implemented by lattice surgery (cutting and joining patches) together with selected *transversal* gates — a *transversal* gate being one applied independently qubit-by-qubit, with no two qubits of a block interacting, so a single physical fault cannot grow into a high-weight error inside the block, which is what makes such gates automatically fault-tolerant. For BB codes, the bicycle architecture of Section 3.4 uses generalized lattice surgery within its logical processing unit, with the same Clifford-friendly structure underneath.

The Clifford group alone is not universal for quantum computation. The *Gottesman-Knill theorem* makes the limitation precise: any quantum circuit using only state preparation in the computational basis, Clifford gates, and computational-basis measurement can be efficiently simulated classically ([Gottesman, 1998](https://arxiv.org/abs/quant-ph/9807006)). Universal computation therefore requires at least one non-Clifford gate. The standard choice is the $T = \mathrm{diag}(1, e^{i\pi/4})$ gate (the $\pi/8$ gate), which together with the Clifford group generates a dense subgroup of $\mathrm{SU}(2^n)$. Implementing $T$ fault-tolerantly is the hard step in universal fault-tolerant computation, and the rest of this section walks through how it is done.

<Accordion>
<AccordionItem title="What does &quot;lattice surgery&quot; actually do? (deeper dive)">

Lattice surgery implements joint logical Pauli measurements by temporarily changing the checks across code patches. For a surface-code logical CNOT with control $C$ and target $T$, one measurement-based description prepares an ancillary logical qubit $A$ in $|+\rangle_L$, measures $\bar{Z}_C\bar{Z}_A$, then $\bar{X}_A\bar{X}_T$, and finally $\bar{Z}_A$, with Pauli-frame updates determined by the three outcomes. A single joint $\bar{Z}_C\bar{Z}_T$ measurement only projects onto a parity subspace; it is not a CNOT. The physical implementation uses repeated syndrome extraction and a layout that maintains error protection throughout the operation. Other logical Clifford operations require related code-deformation or ancillary-state techniques.

In the bicycle architecture, generalized qLDPC surgery and the logical processing unit (LPU) enable selected logical Pauli measurements. Together with the architecture's other instructions, these support logical computation; magic-state injection supplies the non-Clifford resource needed for universality.

- Surface-code surgery: [Horsman et al., 2012](https://arxiv.org/abs/1111.4022).
- BB-specific surgery primitives: [Yoder et al., 2025](https://arxiv.org/abs/2506.03094).

</AccordionItem>
</Accordion>

### 4.2 The $T$ gate via magic-state injection

The surface code does not support a transversal logical $T$. The surface-code and BB architectures discussed here implement it through magic-state injection. The approach is to consume a *magic state* — a logical $|A\rangle = T|+\rangle = (|0\rangle + e^{i\pi/4}|1\rangle)/\sqrt{2}$ prepared in advance — and perform a Clifford-only circuit that effectively applies $T$ to the data qubit. The circuit is shown below as a static diagram; this lesson treats the conditional Clifford correction that follows the ancilla measurement as a *Clifford-frame update*, not as an active mid-circuit correction.

In [6]:
# Static T-gate teleportation / injection circuit.

inj = QuantumCircuit(2, 1)
# q0 = data |psi> (prepared elsewhere); q1 = ancilla magic state |A> = T|+>
inj.h(1)
inj.t(1)  # prepare |A> = T|+> on the ancilla (q1)
inj.barrier()
inj.cx(0, 1)  # CNOT: control = data (q0), target = ancilla (q1)
inj.measure(1, 0)  # measure ancilla; outcome 1 corresponds to an S-frame update on q0

inj.draw("mpl", style="iqp")

<Image src="/learning/images/courses/tools-for-quantum-advantage/error-correction/extracted-outputs/cc93812b-0.avif" alt="Output of the previous code cell" />

The circuit consumes one magic state $|A\rangle$, applies one $\mathrm{CNOT}$ between the data and the ancilla, and measures the ancilla in the computational basis, giving a single classical outcome $m \in \{0, 1\}$. That measurement result determines the branch:

- $m = 0$ branch: the data is in state $T|\psi\rangle$ exactly.
- $m = 1$ branch: the data is in state $T^\dagger|\psi\rangle$, up to a global phase; applying or tracking $S$ gives $ST^\dagger|\psi\rangle = T|\psi\rangle$.

<Accordion>
<AccordionItem title="Why use *T*-state injection? (deeper dive: Eastin-Knill and the Clifford hierarchy)">

The standard recipe used here — fault-tolerant Clifford gates, $T$ supplied via injection — is motivated by two structural results.

**The Clifford hierarchy.** Quantum gates organize into a hierarchy by how they act on Pauli operators under conjugation. Level 1 is the Pauli group; level 2 is the *Clifford group* $\mathcal{C}$, which sends Paulis to Paulis; level 3 contains gates that send Paulis to level-2 elements, and so on.

| Level | Examples | Stabilizer code |
|---|---|---|
| 1 (Pauli) | $X$, $Y$, $Z$ | absorbed into the Pauli/Clifford frame |
| 2 (Clifford) | $H$, $S$, $\mathrm{CNOT}$ | transversal or via lattice surgery |
| 3 | $T$, Toffoli ($\mathrm{CCX}$) | often supplied via injection; transversality depends on the code |

By the Gottesman-Knill theorem, a circuit using only level-2 gates with computational-basis preparation and measurement can be simulated efficiently on a classical computer. Universal quantum computation therefore requires *at least one* non-Clifford resource. $T$ is the canonical choice because it is the simplest such gate and $\{\mathcal{C}, T\}$ is dense in $\mathrm{SU}(2^n)$; nothing in the theory forces $T$ over Toffoli or higher-level operations.

**The Eastin–Knill theorem.** The second result is more constraining. The Eastin-Knill theorem states that a code that detects arbitrary single-qubit errors cannot admit a universal set of *transversal* logical gates ([Eastin and Knill, 2009](https://arxiv.org/abs/0811.4262)). Which gates are transversal depends on the code. The surface-code and BB architectures discussed here use magic-state injection for logical $T$. The transversal set is *not* always the Clifford group: the $[[15,1,3]]$ Reed-Muller code used for distillation in Section 4.3 has a transversal $T$ instead. Eastin-Knill's point is the restriction on a universal transversal set for a fixed code — an additional fault-tolerant mechanism, such as magic-state injection, is needed for universality.

**Why injection resolves this.** Injection implements logical $T$ using Clifford operations, measurements, and a supplied logical magic state $|A\rangle$. Preparing that magic state requires a non-Clifford resource. The state preparation can be done *offline*, decoupled from the moment of consumption. Distillation in Section 4.3 is what brings that offline preparation up to logical-level fidelity.

Similar injection methods apply to other non-Clifford gates. Many fault-tolerant resource estimates use *some* magic state — typically $|A\rangle$ for $T$, $|\mathrm{CCZ}\rangle$ for Toffolis, or higher-order states. The choice of $T$ throughout this lesson is the reference convention.

</AccordionItem>
</Accordion>

The transformation "perform a logical $T$" has now become "supply a high-fidelity logical magic state $|A\rangle$", which is a resource-supply problem. The rest of this section addresses the supply.

### 4.3 Magic-state distillation

It is worth pausing on *why* a separate purification step is needed at all. Everything a fault-tolerant computer does cheaply — Clifford gates and stabilizer measurements — is protected directly by the code. The missing ingredient is the non-Clifford $T$ gate. Magic-state injection sidesteps this by consuming a pre-made $|A\rangle$ state through a Clifford-only circuit, but the difficulty has only moved: the raw injected magic state inherits physical-level error and is far too noisy for a deep logical circuit. A $T$ gate applied via a raw $|A\rangle$ at fidelity $1 - 10^{-2}$ is useless for a logical circuit aiming at $10^{-12}$ logical error.

*Distillation* is the standard way out. It takes many noisy magic states as input and produces fewer high-fidelity ones as output, using only Clifford operations and measurement — operations the underlying code natively supports. The most studied protocol is *15-to-1 distillation* ([Bravyi and Kitaev, 2005](https://arxiv.org/abs/quant-ph/0403025)): fifteen noisy magic states at error probability $p$ yield one output at $\sim 35\,p^3$, using the $[[15,1,3]]$ Reed-Muller code. (The $35$ counts the leading undetected weight-3 error patterns.)

The cubic suppression $p \to 35\,p^3$ is iterated over rounds: starting from $p = 10^{-2}$ raw error, one round gives $\sim 3.5 \times 10^{-5}$; two rounds give $\sim 1.5 \times 10^{-12}$. Each round multiplies the raw-state cost by 15, so reaching deep error targets is expensive in the number of input states required.

In [7]:
# Distillation rounds: error vs raw-state cost. Uses shared imports.
def distill_rounds(p_in, rounds):
    rows, p, raw = [], p_in, 1
    rows.append({"round": 0, "error_estimate": p, "raw_states_per_output": raw})
    for r in range(1, rounds + 1):
        p = 35 * p**3
        raw *= 15
        rows.append({"round": r, "error_estimate": p, "raw_states_per_output": raw})
    return pd.DataFrame(rows)


print(distill_rounds(1e-2, 3).to_string())

   round  error_estimate  raw_states_per_output
0      0    1.000000e-02                      1
1      1    3.500000e-05                     15
2      2    1.500625e-12                    225
3      3    1.182727e-34                   3375


For a surface-code-based fault-tolerant computer, *magic-state factories* — separate code patches running distillation rounds in parallel with the main computation — typically take a substantial fraction of the total physical-qubit budget at the logical error targets of interest (optimized layouts can trim this considerably — [Litinski, 2019](https://arxiv.org/abs/1905.06903)). Higher-yield protocols such as triorthogonal codes ([Bravyi and Haah, 2012](https://arxiv.org/abs/1209.2426)) reduce the cost ratio but do not change the overall structure: distillation is a resource process that runs alongside the main computation and pays for its own qubits.

### 4.4 Distillation on BB codes

The bivariate-bicycle study adds a twist. Because the gross code packs 12 logical qubits into 288 physical, the block's own high-rate logical capacity can be turned on distillation: [Xu et al., 2026](https://arxiv.org/abs/2602.20546) build magic-state factories that run *inside a single BB code block* via qubit recycling, while the bicycle architecture of [Yoder et al., 2025](https://arxiv.org/abs/2506.03094) also supplies non-Clifford resources through dedicated factory modules using surface-code protocols for *cultivation* ([Gidney et al., 2024](https://arxiv.org/abs/2409.17595)) or distillation, depending on the physical-error regime. Either way, magic-state preparation is *not* free — in-block or in a factory, it still costs logical capacity, time, routing, measurement, and decoding; the rate advantage moves and shrinks that cost rather than removing it.

| Scheme | Logical-qubit use | Footprint | Note |
|---|---|---|---|
| Dedicated surface-code factory | separate factory patches | large additional region | factory is spatially separated |
| Single-block factory on gross $[[144,12,12]]$ | reuses some of the 12 logical qubits over time | reduced additional footprint | *not free* — pays in scheduling, time, routing, control |

*Qualitative comparison; see [Xu et al., 2026](https://arxiv.org/abs/2602.20546) and [Yoder et al., 2025](https://arxiv.org/abs/2506.03094) for specifics.*

The $15^r$ growth in input states is one part of the cost: each output magic state at a deep error target consumes hundreds to thousands of raw ones, and those raw states must themselves be produced, stored, and processed somewhere on the chip.

The single-block factory result is specific to the BB / gross-code construction, not a general property of all qLDPC codes — for codes with lower rate, dedicated factories remain natural. The broader point is that the *rate advantage* of high-rate qLDPC codes compounds: the same advantage that reduces memory overhead in Section 3.2 can also lower distillation overhead, since high-rate logical capacity can be brought to bear on it — whether through the single-block factories of Xu et al. 2026 (above) or the dedicated factory modules that Yoder et al. 2025 develop.

### 4.5 What this section establishes — and what it does not prove

The architectures discussed here combine the same building blocks: fault-tolerant Clifford operations, a non-Clifford resource (the $T$ gate, via magic states), and a distillation protocol to bring magic-state fidelity to the level the logical computation requires. Part 4 walked through these primitives in their general form, with one specialization to the BB case study (distillation on high-rate BB codes).

What this section does not prove is that the canonical choices — 15-to-1 distillation, $T$-state injection, lattice surgery for Clifford — are optimal for any specific algorithm. Alternative constructions exist: triorthogonal codes for higher-yield distillation, surgery generalizations, and other non-Clifford resource constructions are active research areas. The case-study choice in this lesson is that the canonical pipeline is a *reference framework* for discussing BB-code resource estimates; quantifying alternatives is its own line of work.

#### Check your understanding

Does magic-state distillation increase or decrease the number of magic states?

<Accordion>
<AccordionItem title="Answer">

Decreases it — many noisy states in, fewer clean ones out (15-to-1: fifteen in, one out per round). It buys *fidelity* with *quantity*.

</AccordionItem>
</Accordion>

#### Check your understanding

Injection already lets you apply a $T$ gate by consuming an $|A\rangle$ state plus a Clifford circuit. So why is distillation still needed?

<Accordion>
<AccordionItem title="Answer">

Because a *raw* $|A\rangle$ prepared from physical qubits is far too noisy — fidelity $\sim 1-10^{-2}$ is useless for a circuit targeting $10^{-12}$ logical error. Distillation spends many noisy $|A\rangle$ plus Clifford-only operations to produce fewer, much cleaner ones ($p \to 35p^3$ per 15-to-1 round). It does not replace the non-Clifford resource — it *supplies* it at usable fidelity. (It consumes Cliffords rather than returning to them; it acts on the magic states, not on data-qubit errors; and it makes preparation longer, not shorter.)

</AccordionItem>
</Accordion>

## 5. Quantum error correction as a quantum-classical system

Sections 3 and 4 treated QEC primarily at the code level — what stabilizer structure a code has, what gates the code supports, what overheads its operations require. A real fault-tolerant computer has to do more than satisfy these code-level requirements: it has to read out syndromes in real time, decode them on classical hardware fast enough to keep up with the QEC cycle, route logical operations between code blocks, and integrate everything into modular hardware that can be fabricated. This section walks through the system-level pieces, with the IBM superconducting case study as the running example. The qubit savings of Part 3 reappear here as classical-compute (decoding) and hardware (connectivity, architecture) cost: the ledger includes the decoder and the wiring, not just the code.

### 5.1 The decoder as the classical half of QEC

A decoder solves a classical inference problem: given a noisy syndrome history, infer a likely recovery class without learning the protected logical state.

When a stabilizer code's checks are measured, the result is a classical bit string called the *syndrome*, indicating which checks fire. The job of the *decoder* is to take this syndrome and infer which error most likely occurred, so that the inferred error can be accounted for in the logical interpretation, usually by updating a classical Pauli/Clifford frame rather than by physically undoing the error in the notebook. The decoder is therefore *classical*: it runs on conventional hardware — FPGAs, ASICs, GPUs, or CPUs — and is part of the quantum-classical loop without which a stabilizer code does not protect any information.

The simplest possible decoder is a *lookup table*: a function from syndromes (bit strings) to most-likely errors (Pauli strings). For codes with very few checks this is trivial. But the lookup-table approach does not scale: the number of entries grows as $2^{m}$ in the number of checks $m$, as the next cell makes concrete.

In [8]:
# Why lookup-table decoding does not scale. Uses shared imports.
rows = [
    {"example": "3-bit repetition", "checks": 2},
    {"example": "Steane [[7,1,3]] (per type)", "checks": 3},
    {"example": "toy surface patch (per type)", "checks": 50},
    {"example": "Gross [[144,12,12]] (total X+Z checks)", "checks": 144},
    {"example": "larger qLDPC block (illustrative, per type)", "checks": 450},
]
df = pd.DataFrame(rows)
df["Look Up Table entries"] = df["checks"].apply(lambda m: f"2^{m}")
df["approx_decimal"] = df["checks"].apply(
    lambda m: f"{2**m:.2e}" if m < 60 else f"~1e{int(m*np.log10(2))}"
)
print(df.to_string())

                                       example  checks Look Up Table entries approx_decimal
0                             3-bit repetition       2                   2^2       4.00e+00
1                  Steane [[7,1,3]] (per type)       3                   2^3       8.00e+00
2                 toy surface patch (per type)      50                  2^50       1.13e+15
3       Gross [[144,12,12]] (total X+Z checks)     144                 2^144          ~1e43
4  larger qLDPC block (illustrative, per type)     450                 2^450         ~1e135


The table above makes the scaling concrete: 3-qubit repetition has 2 checks, fitting 4 entries; toy surface patches have around 50 checks, requiring $2^{50} \approx 10^{15}$ entries; gross-code-scale qLDPC blocks have hundreds of checks, requiring tables far larger than any feasible classical memory. Practical decoders therefore *infer* rather than enumerate — they parameterize the most-likely-error problem and solve it algorithmically.

### 5.2 Decoder families across code families

Surface codes are decoded primarily by *minimum-weight perfect matching* (MWPM): syndrome defects come in pairs at the endpoints of error strings, and the minimum-weight pairing yields the most-likely-error inference. MWPM runs in polynomial time ([blossom algorithm for minimum-weight matching](https://en.wikipedia.org/wiki/Blossom_algorithm)), has been engineered into FPGA implementations achieving real-time latency, and is the mature reference baseline for fault-tolerant prototypes.

Standard minimum-weight matching does not apply directly to bivariate-bicycle codes: a single Pauli error can trigger more than two syndrome bits, so the most-likely-error problem is no longer plain matching. The standard qLDPC decoder is *belief propagation* with *ordered-statistics decoding* (BP + OSD) — BP infers the likely error by passing messages on the code's *Tanner graph* (the bipartite graph with a node for every qubit and every check, an edge wherever a check touches a qubit), and OSD cleans up the cases where BP fails to converge. BP + OSD is accurate but can be too slow for a real-time loop.

*Relay-BP* ([Müller et al., 2025](https://arxiv.org/abs/2506.01779)) is a recent IBM proposal that speeds up BP's convergence to attack exactly this latency. Why speed matters: a superconducting QEC cycle produces a new syndrome roughly every ~1 µs, so if the decoder cannot keep pace *on average*, unprocessed syndromes pile up — the *backlog problem* — and the logical error climbs — better code alone cannot rescue a decoder that falls behind. The binding requirement is therefore not just accuracy but keeping the average decode time within the per-cycle budget; decoder latency is a *systems requirement*. (an FPGA prototype later brought the per-cycle decode latency under this budget — [Maurer et al., 2025](https://arxiv.org/abs/2510.21600)).

### 5.3 Modular architecture: connectivity, couplers, chip lineage

The decoder is the classical half of the system; the *connectivity graph* of the physical qubits is the quantum half. The cost moved out of physical-qubit count in Section 3.2 lands here: BB codes' weight-6 checks include long-range edges that ordinary 2D-nearest-neighbor hardware cannot provide. The chip-level answer is a coupler that reaches across the chip.

![C-couplers enable long-range connections between distant qubits within a quantum chip](/learning/images/courses/tools-for-quantum-advantage/error-correction/c-couplers.gif)

*C-couplers — connectors that couple qubits more distant than their nearest neighbors — enable long-range connections within a single quantum chip. This is the hardware-level answer to the "connectivity" cost flagged in the paragraph above: where the surface code needs only nearest-neighbor couplers, gross-code-style codes require a few longer-range connections — the check graph is a degree-6 layout of two edge-disjoint planar subgraphs, mostly local with some long-range edges. IBM's Quantum Loon processor is an experimental IBM chip incorporating c-couplers, built specifically for proof-of-concept experiments with high-rate qLDPC codes. Source: [fault-tolerance roadmap](https://www.ibm.com/quantum/blog/large-scale-ftqc).*

The c-coupler animation above shows one such hardware response. Other platforms address the same connectivity challenge differently, as Section 5.4 sketches. In IBM's modular architecture, inter-module l-couplers connect separately fabricated chips and support universal adapters between modules. This provides a route to scaling memory and processing without requiring all components on one chip. The fault-tolerance milestones in IBM's published roadmap are summarized below.

| Roadmap year | Chip or system | Capability or milestone | Why it matters for BB |
|---|---|---|---|
| 2025 | Loon | long-range c-coupler demonstrator | tests connectivity for non-local checks |
| 2026 | Kookaburra | qLDPC memory with an attached LPU (target) | combines storage and logical processing |
| 2027 | Cockatoo | inter-module entanglement (target) | connects modules through universal adapters |
| 2028 | Starling proof of concept | magic-state injection with multiple modules (target) | integrates components for universal computation |
| 2029 | Starling | large-scale fault tolerance (target) | scales to 200 logical qubits and 100 million gates |

*Years and milestones follow IBM's published [fault-tolerance roadmap](https://www.ibm.com/quantum/blog/large-scale-ftqc). Target entries describe planned capabilities.*

### 5.4 Other platforms

The systems perspective also clarifies why different hardware platforms naturally lean toward different code families.

| Platform | Strength | Example QEC approaches |
|---|---|---|
| Superconducting | fast gates, mature control | surface baseline; qLDPC via long-range c-couplers |
| Neutral atoms | large arrays, reconfigurable geometry | natural for sparse/non-local (hypergraph-product, HGP-style) layouts |
| Trapped ions | high fidelity, flexible connectivity | small-code demos, modular paths |
| Photonics | networking, measurement-based | fusion-based fault-tolerant architectures |

The matching is not a ranking. Each platform has its own connectivity affordances and dominant error mechanisms, and the code family that fits best there reflects those affordances. Trapped-ion systems can support high-fidelity all-to-all gates within an ion trap, which can facilitate non-local checks. Scaling across traps adds the cost of ion transport or inter-module links. Neutral atoms support reconfigurable geometries — atoms can be moved between operations — which suits sparse non-local codes including hypergraph-product and BB-style constructions. Photonic approaches include fusion-based and measurement-based protocols, in which error syndromes are reconstructed from combinations of measurement outcomes, including fusion outcomes ([Bartolucci et al., 2021](https://arxiv.org/abs/2101.09310)).

The point is not that one of these is the right platform. The point is that the *code-decoder-hardware triple* is what fault-tolerance designs optimize, and different points in that space lead to different sensible code choices. The fault-tolerance bottleneck moves around, but the systems-level structure is platform-independent.

### 5.5 Section summary

What this section establishes is that QEC requires a classical decoder, a connectivity graph that the code's checks can be implemented on, and a system architecture that integrates both with the quantum hardware. The IBM superconducting case study (the chip lineage tabulated in Section 5.3) gives a concrete sequence of milestones for that integration on one platform.

What this section does not establish is that the IBM roadmap is the only viable system architecture — neutral atoms, ions, and photonics offer parallel paths — or that the timeline holds, since whether each chip ships as planned is an engineering execution question rather than a code-theory result. It also does not establish that current decoders are sufficient at the target latency: Relay-BP and follow-up proposals are recent and not yet hardware-deployed at scale. The system view reframes each of these as *open problems with concrete benchmark targets* rather than ambient uncertainties.

#### Check your understanding

Lookup-table decoding scales as $2^{(\text{number of checks})}$. It is fine for the 3-qubit repetition code (2 checks). Why does it fail for the gross code ($\sim 144$ checks), and what replaces it?

<Accordion>
<AccordionItem title="Answer">

$2^{144} \approx 10^{43}$ table entries — far beyond any table one could feasibly store. So practical decoders *infer* the most likely error algorithmically instead of enumerating syndromes: MWPM for the surface code, BP+OSD for BB, Relay-BP for real-time qLDPC. (The syndrome is still a classical bit string — the obstacle is the exponential *size* of the syndrome space, not its type.)

</AccordionItem>
</Accordion>

#### Check your understanding

Consider a superconducting QEC cycle that yields a new syndrome every 1 µs. Why must the decoder keep pace *on average*, and what breaks if it cannot?

<Accordion>
<AccordionItem title="Answer">

The decoder must process syndrome data at least as fast as it arrives. Otherwise, a backlog grows and operations that depend on decoded outcomes must wait. Throughput controls this backlog; decoding latency also matters for real-time feedforward. Relay-BP is designed for fast, parallel FPGA or ASIC implementations ([Müller et al., 2025](https://arxiv.org/abs/2506.01779)), and subsequent FPGA work demonstrates real-time gross-code memory decoding ([Maurer et al., 2025](https://arxiv.org/abs/2510.21600)). Decoding cannot always be deferred until the end because later quantum instructions can depend on earlier measurement outcomes.

</AccordionItem>
</Accordion>

## 6. Quantum error correction in the framework of quantum advantage

[Lanes et al., 2025](https://arxiv.org/abs/2506.20658) ground advantage in two criteria — a result must be rigorously *validated*, and it must show a genuine *quantum separation* over classical methods. Operationally, the accuracy spectrum lets us turn those into four concrete questions for any claim:

1. **What is the task?** Sampling problems, expectation-value computations, search, structured simulation, and analog problems all have different correctness criteria.

2. **How is correctness validated?** Correctness must be rigorously validated, while performance is assessed against the best available classical methods. Validation can use classical verification, cross-checks between methods, or structural properties such as peakedness in sampling.

3. **What resource is paid?** Samples, qubits, decoding compute, runtime, connectivity, calibration time, classical post-processing — each tool on the spectrum pays a different resource.

4. **Does QEC preserve or erase the quantum separation after overhead?** Even when QEC works as intended, the overhead it adds may shrink the separation between quantum and classical algorithms more than the algorithmic speedup buys. For deep circuits at stringent logical-error targets, QEC is the standard route to scalable error control; for shallow circuits or near-term advantage windows, suppression-plus-mitigation or post-selected detection may give a better effective separation.

The framing is that *these tools form a resource menu, not a linear recipe*. They compose: a real workload chooses a subset of the menu, and the validation-and-separation analysis is what tells us whether the subset closes the gap to advantage.

### The cost crossover

The cost shapes of different accuracy-spectrum tools are qualitatively different. For the schemes emphasized in earlier lessons, mitigation and post-selected detection can have costs that grow exponentially with circuit volume or total error rate: sampling overhead in probabilistic error cancellation (PEC) grows roughly as $\exp(c \cdot V \cdot p)$, and post-selected detection acceptance decays roughly as $\exp(-c' \cdot V \cdot p)$, where $V$ is circuit volume and $p$ is physical error rate. This schematic model assumes approximately independent faults; the constants depend on the protocol and check coverage. Fault-tolerant correction with decoding has a different shape: a large upfront architectural overhead — code patches, magic-state factories, decoders, modular hardware — followed by more controlled scaling with circuit volume once logical error rates are suppressed. These cost shapes are schematic, with the exact crossover point depending on physical error rates, target logical fidelity, and the specific algorithm. The qualitative point is that mitigation and detection can be the right choice at smaller volumes, while correction can become favorable once circuit volume and reliability targets make those lighter-weight approaches too costly; the crossover defines the *regime* in which any advantage claim sits.

For near-term advantage demonstrations, mitigation and post-selection can be useful when their cost stays manageable at the volumes current hardware can run. For large-scale fault-tolerant computation, decoding-based correction becomes central because the overhead of rigorous mitigation or post-selected detection alone can become prohibitive. These methods can also be combined along the QEC continuum ([QEC continuum blog](https://www.ibm.com/quantum/blog/qec-continuum)).

#### Check your understanding

In the schematic model above, PEC, post-selection, and fault-tolerant correction with decoding have *qualitatively different* cost shapes as functions of circuit volume $V$. Describe each shape, and what it implies for when each tool wins.

<Accordion>
<AccordionItem title="Answer">

In this model, PEC and post-selection have exponential costs in volume — PEC sampling overhead $\sim e^{cVp}$, post-selection acceptance $\sim e^{-c'Vp}$. Fault-tolerant correction with decoding instead pays a large *upfront* architectural cost (code patches, magic-state factories, decoders, modular hardware) and then grows *gently* once logical error rates are suppressed. The relative costs can cross: small circuits favor mitigation/detection, deep circuits favor correction, and that crossover defines the *regime* of any advantage claim. (Real cost models are messier; the robust point is the difference in shape, not an exact crossing. Note that PEC and post-selection costs in this model are *not* flat in $V$ — their blow-up is the whole point — and correction is specifically *not* exponential in $V$.)

</AccordionItem>
</Accordion>

#### Where this leaves the "when" question

The honest endpoint is not that correction *wins* — or that mitigation or detection wins — but that *which tool fits a given advantage claim* now turns on stated, checkable things: the task, how correctness is validated, and the resource budget. The case study in Sections 3–5 is one concrete attempt to push the correction end of the spectrum until that cost crossover moves in its favor, and it is still moving. The most recent step, *code concatenation*, builds a larger code by encoding the logical qubits of an *inner* code into the data slots of an *outer* code, relaxing the overhead–suppression–connectivity tradeoff one notch further by supplying non-locality through the inner code's *logical* operations rather than through hardware — reported to reach the teraquop regime for memory with lower space overhead than the 288-data-qubit two-gross code in the authors' analysis ([Wills et al., 2026](https://arxiv.org/abs/2605.21898)). The large-scale BB architecture is still supported mainly by circuit-level simulations and architectural analysis. Separately, IBM and partners reported quantum-advantage demonstrations on 30 July 2026 ([quantum-advantage announcement](https://www.ibm.com/quantum/blog/quantum-advantage)), with ongoing comparisons against classical methods through the Quantum Advantage Tracker. The experiment by IBM and the University of Chicago used spacetime-code post-selection, the approach introduced in the previous lesson, and reported roughly a $10\times$ reduction in effective gate error for a 70-logical-qubit computation ([quantum-advantage announcement](https://www.ibm.com/quantum/blog/quantum-advantage)). Starling's large-scale fault-tolerance milestone remains a 2029 target ([fault-tolerance roadmap](https://www.ibm.com/quantum/blog/large-scale-ftqc)). That is what [Lanes et al., 2025](https://arxiv.org/abs/2506.20658) anticipate — advantage arriving not as a single milestone but as a *sequence* of increasingly robust demonstrations, each validated against the best classical baseline and settled in the open.

#### Evaluating new results

Choosing an error-handling strategy starts with the task, how correctness will be validated, and the full resource cost. The code, decoder, and hardware examples in this lesson help assess whether error correction can meet those requirements while preserving a computational advantage. Mitigation and post-selection can complement error correction, including on logical circuits ([QEC continuum blog](https://www.ibm.com/quantum/blog/qec-continuum)).

The [Quantum Advantage Tracker](https://www.ibm.com/quantum/blog/quantum-advantage-tracker) supports open comparison of candidate quantum computations with leading classical methods. As both improve, performance comparisons and resource estimates need to be revisited. For each new result, ask what accuracy is demonstrated, which resources are counted, and whether the comparison uses the best available classical methods. Use these questions when you assess or design an error-handling workflow.

### References

**Quantum advantage framework**

1. Lanes et al., "A Framework for Quantum Advantage," [arXiv:2506.20658](https://arxiv.org/abs/2506.20658) (2025).

**Quantum LDPC codes and the bicycle architecture**

2. Bravyi, Cross, Gambetta, Maslov, Rall, Yoder, "High-threshold and low-overhead fault-tolerant quantum memory," Nature **627**, 778 (2024). [Publisher page](https://www.nature.com/articles/s41586-024-07107-7).

3. Yoder et al., "Tour de gross: a modular quantum computer based on bivariate bicycle codes," [arXiv:2506.03094](https://arxiv.org/abs/2506.03094) (2025).

4. Xu, Liu, Rall, He, Ding, "Distilling Magic States in the Bicycle Architecture," [arXiv:2602.20546](https://arxiv.org/abs/2602.20546) (2026).

5. Wills et al., "Concatenating Algebraic Codes over High-Rate Quantum LDPC Codes," [arXiv:2605.21898](https://arxiv.org/abs/2605.21898) (2026).

6. Tillich and Zémor, "Quantum LDPC codes with positive rate and minimum distance proportional to $n^{1/2}$," [arXiv:0903.0566](https://arxiv.org/abs/0903.0566) (2009).

7. Panteleev and Kalachev, "Asymptotically Good Quantum and Locally Testable Classical LDPC Codes," [arXiv:2111.03654](https://arxiv.org/abs/2111.03654) (2022).

8. Leverrier and Zémor, "Quantum Tanner codes," [arXiv:2202.13641](https://arxiv.org/abs/2202.13641) (2022).

**Surface codes and lattice surgery**

9. Fowler, Mariantoni, Martinis and Cleland, "Surface codes: towards practical large-scale quantum computation," [arXiv:1208.0928](https://arxiv.org/abs/1208.0928) (2012).

10. Bombín and Martín-Delgado, "Optimal resources for topological 2D stabilizer codes," [arXiv:quant-ph/0703272](https://arxiv.org/abs/quant-ph/0703272) (2007).

11. Bravyi and Kitaev, "Quantum codes on a lattice with boundary," [arXiv:quant-ph/9811052](https://arxiv.org/abs/quant-ph/9811052) (1998).

12. Kitaev, "Fault-tolerant quantum computation by anyons," [arXiv:quant-ph/9707021](https://arxiv.org/abs/quant-ph/9707021) (2003).

13. Horsman, Fowler, Devitt and Van Meter, "Surface code quantum computing by lattice surgery," [arXiv:1111.4022](https://arxiv.org/abs/1111.4022) (2012).

14. Fowler and Gidney, "Low overhead quantum computation using lattice surgery," [arXiv:1808.06709](https://arxiv.org/abs/1808.06709) (2018).

**Magic states and fault-tolerant computation**

15. Bravyi and Kitaev, "Universal quantum computation with ideal Clifford gates and noisy ancillas," [arXiv:quant-ph/0403025](https://arxiv.org/abs/quant-ph/0403025) (2005).

16. Bravyi and Haah, "Magic-state distillation with low overhead," [arXiv:1209.2426](https://arxiv.org/abs/1209.2426) (2012).

17. Litinski, "Magic State Distillation: Not as Costly as You Think," [arXiv:1905.06903](https://arxiv.org/abs/1905.06903) (2019).

18. Gidney, Shutty and Jones, "Magic state cultivation," [arXiv:2409.17595](https://arxiv.org/abs/2409.17595) (2024).

**Decoding**

19. Müller et al., "Improved belief propagation is sufficient for real-time decoding of quantum memory," [arXiv:2506.01779](https://arxiv.org/abs/2506.01779) (2025).

20. Maurer et al., "Real-time decoding of the gross code memory with FPGAs," [arXiv:2510.21600](https://arxiv.org/abs/2510.21600) (2025).

**Code foundations and other architectures**

21. Bravyi, Poulin and Terhal, "Tradeoffs for reliable quantum information storage in 2D systems," Phys. Rev. Lett. **104**, 050503 (2010). [arXiv:0909.5200](https://arxiv.org/abs/0909.5200).

22. Eastin and Knill, "Restrictions on transversal encoded quantum gate sets," Phys. Rev. Lett. **102**, 110502 (2009). [arXiv:0811.4262](https://arxiv.org/abs/0811.4262).

23. Bartolucci et al., "Fusion-based quantum computation," [arXiv:2101.09310](https://arxiv.org/abs/2101.09310) (2021).

24. Gottesman, "The Heisenberg Representation of Quantum Computers," [arXiv:quant-ph/9807006](https://arxiv.org/abs/quant-ph/9807006) (1998).

**IBM Quantum resources**

25. IBM Quantum blog, "How IBM will build the world's first large-scale, fault-tolerant quantum computer." https://www.ibm.com/quantum/blog/large-scale-ftqc

26. IBM Quantum blog, "Landmark IBM error correction paper published on the cover of Nature." https://www.ibm.com/quantum/blog/nature-qldpc-error-correction

27. IBM Quantum blog, "Researchers demonstrate quantum advantage through trusted quantum computation" (30 July 2026). https://www.ibm.com/quantum/blog/quantum-advantage

28. IBM Quantum blog, "Quantum Advantage Tracker: the race to advantage." https://www.ibm.com/quantum/blog/quantum-advantage-tracker

29. IBM Quantum blog, "From error mitigation to fault-tolerant quantum computing." https://www.ibm.com/quantum/blog/qec-continuum

30. IBM Quantum blog, "Qiskit Paulice: postselected quantum error correction for near-term hardware." https://www.ibm.com/quantum/blog/qiskit-paulice